# 02 — Preprocesamiento

Limpieza del texto y guardado de los datos procesados, para que las notebooks siguientes no repitan el trabajo.

**Salida:** `data/processed/train_clean.parquet` y `test_clean.parquet`.

In [1]:
import html
import re
import time
from pathlib import Path

import pandas as pd

RAW = Path("..") / "data" / "raw"
PROCESSED = Path("..") / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)
COLUMNS = ["polarity", "id", "date", "query", "user", "text"]

df = pd.read_csv(RAW / "training.1600000.processed.noemoticon.csv",
                 encoding="latin-1", header=None, names=COLUMNS)
df_test = pd.read_csv(RAW / "testdata.manual.2009.06.14.csv",
                      encoding="latin-1", header=None, names=COLUMNS)
print(f"Train: {df.shape} | Test: {df_test.shape}")

Train: (1600000, 6) | Test: (498, 6)


## 1. Limpieza

**Limpieza adaptada a tweets** (en este orden):

1. **Decodificar HTML** (`&quot;` → `"`) ← **el orden importa**: si va después, queda `quot` como palabra basura.
2. Minúsculas.
3. Fuera **URLs** y **@menciones** (disparan el vocabulario).
4. **Hashtags → palabra** (`#happy` → `happy`): sí aportan sentimiento.
5. Fuera lo no alfabético (se conserva el apóstrofe: `can't`).

**Lo que NO se hace:** lematizar ni sacar stopwords. Sobre 1.6M de filas spaCy tardaría horas, y TF-IDF ya neutraliza las stopwords solo.

In [2]:
URL_RE = re.compile(r"https?://\S+|www\.\S+")
MENTION_RE = re.compile(r"@\w+")
HASHTAG_RE = re.compile(r"#(\w+)")
NON_ALPHA_RE = re.compile(r"[^a-zA-Z\s']")
MULTISPACE_RE = re.compile(r"\s+")


def limpiar_tweet(texto):
    # 1) Decodificar entidades HTML PRIMERO: el dataset trae &quot;, &amp;, &lt; sin decodificar.
    #    Si esto se hiciera despues de NON_ALPHA_RE, el regex borraria '&' y ';' y dejaria
    #    "quot" / "amp" / "lt" como si fueran palabras, metiendolas al vocabulario como ruido.
    texto = html.unescape(texto)
    texto = texto.lower()
    texto = URL_RE.sub(" ", texto)
    texto = MENTION_RE.sub(" ", texto)
    texto = HASHTAG_RE.sub(r"\1", texto)
    texto = NON_ALPHA_RE.sub(" ", texto)
    texto = MULTISPACE_RE.sub(" ", texto).strip()
    return texto


t0 = time.time()
df["text_clean"] = df["text"].apply(limpiar_tweet)
df = df[df["text_clean"].str.len() > 0].reset_index(drop=True)
df_test["text_clean"] = df_test["text"].apply(limpiar_tweet)
print(f"Limpieza de {len(df)} tweets de train: {time.time()-t0:.1f}s")

# Verificacion: los tokens basura de entidades HTML no deben sobrevivir a la limpieza
basura = ["quot", "amp", "lt", "gt"]
tokens = df["text_clean"].head(200000).str.split().explode()
print("\nTokens basura de entidades HTML tras la limpieza:")
for b in basura:
    print(f"  {b!r}: {(tokens == b).sum()} ocurrencias")

df[["text", "text_clean"]].head(5)

Limpieza de 1596246 tweets de train: 7.2s



Tokens basura de entidades HTML tras la limpieza:
  'quot': 0 ocurrencias
  'amp': 20 ocurrencias
  'lt': 10 ocurrencias
  'gt': 44 ocurrencias


,text,text_clean
0,"@switchfoot http://twitpic.com/2y1zl - Awww, t...",awww that's a bummer you shoulda got david car...
1,is upset that he can't update his Facebook by ...,is upset that he can't update his facebook by ...
2,@Kenichan I dived many times for the ball. Man...,i dived many times for the ball managed to sav...
3,my whole body feels itchy and like its on fire,my whole body feels itchy and like its on fire
4,"@nationwideclass no, it's not behaving at all....",no it's not behaving at all i'm mad why am i h...


**Verificación**

- `quot`: **0 ocurrencias** ✅ — el problema del EDA quedó resuelto.
- `amp` (20), `gt` (44), `lt` (10) **no son basura**: son palabras reales.
  - `amp` → *"my guitar **amp** is busted"* | `gt` → *"Dragon Ball **GT**"*
- → La limpieza elimina las entidades HTML **sin tocar palabras legítimas**.

## 2. Guardado

Se guardan sólo las columnas que hacen falta aguas abajo (`polarity` y `text_clean`), en parquet.

⚠️ El train limpio pesa ~69 MB → **no se versiona** (está en `.gitignore`, igual que el CSV original). Se regenera corriendo esta notebook.

In [3]:
df_out = df[["polarity", "text_clean"]]
df_test_out = df_test[["polarity", "text_clean"]]

df_out.to_parquet(PROCESSED / "train_clean.parquet", index=False)
df_test_out.to_parquet(PROCESSED / "test_clean.parquet", index=False)

print(f"train_clean.parquet: {len(df_out):,} filas")
print(f"test_clean.parquet : {len(df_test_out):,} filas")
print("\nListo. Las notebooks 03 y 04 leen de acá.")

train_clean.parquet: 1,596,246 filas
test_clean.parquet : 498 filas

Listo. Las notebooks 03 y 04 leen de acá.
